In [ ]:
import snscrape.modules.twitter as twitter_scrapper
import pandas as pd
import numpy as np
import regex as re
import matplotlib as plt
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer

#### Data Extraction

In [3]:
scraped_data = []

handles = ["joejonas", "Nike", "TheOnion", "netflix" , "LFC" , "UNICEF"]
for account_name in handles: 
    query = "from:" + account_name
    i = 0
    for tweet in twitter_scrapper.TwitterSearchScraper(query).get_items():
        if i>=1200:
            break
        scraped_data.append([tweet.date, tweet.url, tweet.username, tweet.content])

        i += 1

column_labels = ["date", "tweet_url" , "username", "tweet"]
tweets_df = pd.DataFrame(scraped_data, columns=column_labels)


tweets_df.to_csv("data.csv")

#### Preprocessing

In [77]:
tweets_df = pd.read_csv("data.csv" , index_col=0)

In [78]:
tweets_df['tweet'] = tweets_df['tweet'].str.lower()
tweets_df['tweet'] = tweets_df['tweet'].str.replace(r'http\S+', '', regex=True)
tweets_df['tweet'] = tweets_df['tweet'].str.replace(r'[^0-9a-zA-Z\s]+', '', regex=True)
tweets_df['tweet'] = tweets_df['tweet'].str.replace('\d+', '')
tweets_df['tweet'] = tweets_df['tweet'].replace(r'\n+|\t+','', regex=True)
tweets_df['tweet'] = tweets_df['tweet'].str.replace(r'\s+', ' ', regex=True)
tweets_df['tweet'] = tweets_df['tweet'].str.strip()

# Reading Stop Words
file = open("./stop_words.txt", "r", encoding="utf8")
stop_words = file.read()
stop_words = stop_words.split()

tweets_df['tweet'] = tweets_df['tweet'].apply(lambda x: ' '.join([word for word in x.split() if word not in (stop_words)]))

tweets_df['tweet'].replace('', np.nan, inplace=True)
tweets_df.dropna(subset=['tweet'], inplace=True)
tweets_df.index = np.arange(len(tweets_df))

# tweets_df.username = pd.Categorical(tweets_df.username)
# tweets_df['label'] = tweets_df.username.cat.codes

c:\Users\Muhammad Shahmeer\AppData\Local\Programs\Python\Python37\lib\site-packages\ipykernel_launcher.py:4: FutureWarning: The default value of regex will change from True to False in a future version.
  after removing the cwd from sys.path.


In [7]:
train_tweets, test_tweets = train_test_split(tweets_df, test_size=0.2, shuffle=True)

train_tweets.index = np.arange(len(train_tweets))
test_tweets.index = np.arange(len(test_tweets))

#### Creating Vocab

In [8]:
def create_vocab(df):
    vocab = list(df['tweet'].str.split(' ', expand=True).stack().unique())
    vocab = list(set(vocab))
    return vocab

vocab = create_vocab(train_tweets)

#### Bag of Word

In [9]:
def vectorize(vocab, sentence):
    vector=[]
    for w in vocab:
        vector.append(sentence.count(w))
    return vector

def bag_of_words(df):
    bow = df.apply(lambda row: vectorize(vocab, row.str.split()['tweet']), axis=1)
    df['bag_of_words'] = bow

bag_of_words(train_tweets)
bag_of_words(test_tweets)

#### Creating Embeddings for each tweets

In [24]:
model = SentenceTransformer('all-MiniLM-L6-v2')

train_tweets["embeddings"] = list(model.encode(train_tweets["tweet"]))
test_tweets["embeddings"] = list(model.encode(test_tweets["tweet"]))

In [32]:
train_tweets.columns

Index(['date', 'tweet_url', 'username', 'tweet', 'bag_of_words', 'embeddings'], dtype='object')

#### KNN

#### Using Embedding

In [27]:
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix



def scikit_Knn (k, distance_metric, f_train, l_train, f_test):
  knn = KNeighborsClassifier(n_neighbors=k, p=distance_metric)
  knn.fit(f_train, l_train)
  predicted_label = knn.predict(f_test)

  return predicted_label

def scikit_eval(l_valid, predicted_label):
  classes = list(range(0,10))
  acuracy = accuracy_score(l_valid, predicted_label, normalize=False)

  evaluation_report = classification_report(l_valid, predicted_label, target_names=classes)
  f1_score = evaluation_report['macro avg']['f1-score']

  cm = confusion_matrix(l_valid, predicted_label, labels=classes)

  return (acuracy, f1_score, cm)


def mfold_scikit(m, k, distance_metric, features, labels):
  kf = KFold(n_splits=m)
  p_labels = list()
  for train_index, test_index in kf.split(features):
    # print("TRAIN:", train_index, "TEST:", test_index)
    # print("TRAIN:", len(train_index), "TEST:", len(test_index))
    f_train, f_valid = features[train_index], features[test_index]
    l_train, l_valid = labels[train_index], labels[test_index]

    predicted_label = scikit_Knn(k, distance_metric, f_train, l_train, f_valid)
    # print(predicted_label.shape)
    p_labels.append(predicted_label.tolist())

  p_labels = sum(p_labels, [])
  p_labels = np.array(p_labels)
  return p_labels

In [104]:
train_tweets.username = pd.Categorical(train_tweets.username)
train_tweets['label'] = train_tweets.username.cat.codes

train_features = list(train_tweets["embeddings"])
len(train_features[1])

train_labels = train_tweets["username"]

knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(train_features, train_labels)
# predicted_label = knn.predict(f_test)

c:\Users\Muhammad Shahmeer\AppData\Local\Programs\Python\Python37\lib\site-packages\sklearn\utils\validation.py:746: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  array = np.asarray(array, order=order, dtype=dtype)


ValueError: Expected 2D array, got 1D array instead:
array=[1
 array([ 5.94160818e-02,  8.79166368e-03,  4.94763665e-02, -6.97350800e-02,
         1.66815579e-01,  5.50480094e-03,  3.31854895e-02,  3.31453905e-02,
         1.32781570e-03,  8.30778703e-02, -3.64511199e-02, -5.05473278e-02,
         1.46177839e-02,  1.67986676e-02,  3.39365043e-02, -2.55810805e-02,
        -5.30451685e-02, -6.90488964e-02, -6.20594807e-02, -2.84692906e-02,
        -6.00297451e-02, -4.76543680e-02, -1.12791518e-02,  3.95151787e-02,
        -8.30068961e-02, -2.20191013e-02,  4.95487489e-02,  7.90022761e-02,
        -7.79097201e-03, -1.42159802e-03,  7.72533612e-03,  6.29931130e-03,
         1.17949760e-02,  4.76024114e-02,  2.76996940e-02, -5.22514153e-03,
        -6.37852848e-02, -6.22656345e-02, -7.00972974e-02,  9.05615315e-02,
        -1.24701457e-02, -6.97073266e-02,  5.25878668e-02,  3.61186862e-02,
        -5.36830339e-04,  1.82485748e-02,  3.07820588e-02, -3.36483195e-02,
         3.01976651e-02,  